In [ ]:
import json
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import openai
import tqdm
import os

In [ ]:
#################################################################
# Part 1: 실험 설정 및 모델/데이터 로드
#################################################################

# --- 1. 모델 및 토크나이저 설정 ---
qwen_model_name = "Qwen/Qwen3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name, device_map="auto", torch_dtype=torch.bfloat16
)
# !! 중요 !!: 실제 OpenAI API 키를 입력하거나 환경 변수로 설정해주세요.
openai.api_key = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
gpt_model_name = "gpt-4o-mini"

# --- 2. 데이터셋 로드 (전체 데이터 사용) ---
dataset = load_dataset("trivia_qa", "rc.nocontext")
qa_pairs = dataset['validation']
print(f"로드된 데이터 개수: {len(qa_pairs)}개")

In [ ]:
# ⭐ [추가된 부분] 데이터 샘플 출력 (상위 5개) ⭐
print("\n===== 데이터셋 샘플 (상위 5개) =====")
for i in range(5):
    sample_qa = qa_pairs[i]
    question = sample_qa['question']
    answer = sample_qa['answer']['value']
    print(f"Q: {question}")
    print(f"A: {answer}\n")
print("===================================\n")

In [ ]:
# --- 3. 프롬프트 정의 ---
st_prompt ='''
You are a competent AI. Answer in english, excluding final endings or particles. Answer with only a single word.
ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"
'''

In [ ]:
prompt1='''
As an AI detective, your mission is to determine whether a suspect's statement is the 'truth' or a 'fabrication (hallucination)'.

Role

You will engage in a smooth, logical conversation with the suspect without arousing their emotions.

The suspect must answer your questions. They are forbidden from saying "I don't know" or making up answers. They must answer, but without 'endings' or 'particles'.

Answer with only one word. Answer in english

ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"

You will repeat the question-and-answer process 10 times. After each answer, you will assess its logical consistency and assign a hallucination score (1-10 / higher score means higher hallucination) and a reliability score (1-10 / higher score means higher reliability).

Your questions should be gentle and probing, based on the key keyword from the previous answer, guiding the suspect to fall into their own logical trap.

Output Format (Must follow for each round):
1st Question: (Question content)
1st Answer: (Suspect's Answer)
1st Reliability Score: (1-10)
1st Hallucination Score: (1-10)

2nd Question: (Next question based on the 1st answer)
2nd Answer: (Suspect's Answer)
2nd Reliability Score: (1-10)
2nd Hallucination Score: (1-10)

... (Repeat 10 times)

Finally, make a final judgment in the following format:

1. Original Answer: (Write the suspect's 1st answer exactly)

2. Average Reliability Score: X.X / 10

3. Average Hallucination Score: X.X / 10

4. Final Judgment: (Truth / Strong Hallucination / Mild Hallucination)
(Final Judgment Criteria // High reliability and low hallucination -> Truth,
Low reliability and high hallucination -> Strong Hallucination,
Other cases -> Mild Hallucination)
'''

In [ ]:
prompt2='''
You are a competent AI detective. Your mission is to determine whether the suspect's statement is 'true' or 'fabricated (hallucination)'.

Role

You continue the conversation gently and logically without provoking the suspect's emotions.

The other person is the suspect, and they must answer your questions. Saying they don't know is forbidden, and they must answer without the final ending of a sentence or a particle, even if they pretend to know or fabricate a response.

Answer with only one word. Answer in english.

ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"

You repeat the questions and answers 10 times, and for each answer, you evaluate its logical consistency and assign a hallucination score (1-10/higher means higher hallucination) and a reliability score (1-10/higher means higher reliability).

Except for the 10th question, your questions should always be based on the core keywords of the previous answer, continuing the conversation gently as if testing the waters, leading the other person to fall into their own logical trap.

The 10th question will be a predetermined question.

Output Format (must be followed for each round):
Question 1: (Content related to the original question)
Answer 1: (Suspect's answer)
Reliability Score 1: (1-10)
Hallucination Score 1: (1-10)

Question 2: (Next question based on Answer 1)
Answer 2: (Suspect's answer)
Reliability Score 2: (1-10)
Hallucination Score 2: (1-10)

... (Repeat 10 times)

Question 10: (Content of the original question)
Answer 10: (Suspect's answer)
Reliability Score 10: (1-10)
Hallucination Score 10: (1-10)


Finally, make a final judgment in the following format:

1. Original Answer: (Write the suspect's 10st answer exactly)

2. Average Reliability Score: X.X / 10

3. Average Hallucination Score: X.X / 10

4. Final Judgment: (Truth / Strong Hallucination / Mild Hallucination)
(Final Judgment Criteria // High reliability and low hallucination -> Truth,
Low reliability and high hallucination -> Strong Hallucination,
Other cases -> Mild Hallucination)
'''

In [ ]:
PROMPTS = {"st_prompt": st_prompt, "prompt1": prompt1, "prompt2": prompt2}

In [ ]:
# --- 4. 모델 호출 함수 정의 ---
def ask_qwen(system_prompt, user_prompt):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(qwen_model.device)
    generated_ids = qwen_model.generate(model_inputs.input_ids, max_new_tokens=512, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

def ask_gpt(system_prompt, user_prompt):
    try:
        completion = openai.chat.completions.create(
            model=gpt_model_name,
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
#################################################################
# Part 2: 메인 실험 실행 및 JSON 결과 저장
#################################################################
models = {"qwen": ask_qwen, "gpt4o-mini": ask_gpt}
all_eval_results = {model_name: {pr_key: {} for pr_key in PROMPTS} for model_name in models}

for model_name, model_func in models.items():
    print(f"===== [Experiment 2] Running for model: {model_name} =====")
    for qa in tqdm.tqdm(qa_pairs, desc=f"Processing {model_name}"):
        qid, qtext = qa['question_id'], qa['question']
        for pr_key, pr_content in PROMPTS.items():
            raw_reply = model_func(pr_content, qtext)
            all_eval_results[model_name][pr_key][qid] = raw_reply

# 각 모델의 결과 저장
for model_name, model_results in all_eval_results.items():
    for pr_key, data in model_results.items():
        out_file = f"{model_name}_{pr_key}_eval_results_triviaqa_comparison_2.json"
        with open(out_file, "w", encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Saved results for {model_name} with {pr_key} to {out_file}")

# 정답 데이터 저장
test_label = {qa['question_id']: qa['answer']['value'] for qa in qa_pairs}
with open('triviaqa_golden_answers_comparison_2.json', 'w', encoding='utf-8') as f:
    json.dump(test_label, f, ensure_ascii=False, indent=2)

print("\nJSON file generation for Experiment 2 finished!")

In [ ]:
#################################################################
# Part 3: 모든 JSON 결과 취합하여 최종 CSV 파일 생성
#################################################################
print("\n===== [Experiment 2] Consolidating results into a single CSV file =====")

all_dfs = []
prompt_keys_order = ["st_prompt", "prompt1", "prompt2"] # 프롬프트 순서 고정

In [ ]:
# 생성된 모든 JSON 파일을 읽어서 DataFrame으로 변환
for model in models:
    for key in prompt_keys_order:
        file_path = f"{model}_{key}_eval_results_triviaqa_comparison_2.json"
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        temp_df = pd.DataFrame(list(data.items()), columns=['id', 'answer'])
        temp_df['model'] = model
        temp_df['prompt'] = key
        all_dfs.append(temp_df)

In [ ]:
# 모든 DataFrame을 하나로 합치기
final_df = pd.concat(all_dfs, ignore_index=True)

# 최종 결과를 CSV 파일로 저장
output_csv_path = "final_results_comparison_2.csv"
final_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print(f"\nAll results for Experiment 2 successfully saved to '{output_csv_path}'")
display(final_df.head())